### **Cuaderno14 MCC225: Tarea individual para la Segunda Exposición Académica Programada**

#### **Evaluación experimental de sistemas multimodales con datos reales**

Este cuaderno es una base de trabajo individual. Cada estudiante debe resolverlo, ejecutarlo con datos reales, guardar los resultados en su repositorio y usar sus hallazgos como soporte para la Segunda Exposición Académica Programada.

El objetivo no es hacer una demostración visual, sino construir una evaluación reproducible de un sistema multimodal. El trabajo conecta BERT, VisualBERT, UNITER, ViLT, BLIP, BLIP-2, LLaVA, CoCa, DALL E 2 y modelos de difusión latente desde una pregunta experimental común: ¿el sistema alinea correctamente imagen y texto, cuándo falla y cómo se documenta ese fallo?-

### **Reglas del trabajo individual**

#### **Condiciones mínimas**

El trabajo es individual. Cada estudiante debe mantener este cuaderno resuelto en su propio repositorio. No se acepta únicamente una exposición oral sin evidencia ejecutable.

El repositorio debe contener como mínimo:

```text
.
├── notebooks/
│   └── Cuaderno14_MCC225.ipynb
├── data/
│   ├── raw/
│   └── processed/
├── outputs/
│   ├── figures/
│   ├── tables/
│   └── metrics/
├── reports/
│   └── reporte_exposicion_2.md
├── requirements.txt
└── README.md
```

El estudiante debe entregar resultados reales, no capturas aisladas. El cuaderno debe producir archivos de métricas, tablas de errores, figuras y un reporte técnico breve.

### **Qué se debe demostrar**

#### Preguntas experimentales

El trabajo debe responder estas preguntas:

1. ¿Qué tan bien alinea el modelo imágenes y textos reales?
2. ¿Qué cambia al degradar la imagen o perturbar los captions?
3. ¿Qué errores aparecen con mayor frecuencia?
4. ¿Qué métrica captura mejor cada tipo de comportamiento?
5. ¿Qué limitaciones quedan fuera de las métricas automáticas?
6. ¿El experimento es reproducible por otra persona?

La exposición debe sostenerse en evidencia del repositorio: métricas, tablas, ejemplos correctos, ejemplos fallidos y discusión de limitaciones.

### **Instalación sugerida**

#### **Dependencias**

Ejecute esta celda solo si el entorno no tiene las bibliotecas requeridas. En un repositorio serio, la misma lista debe aparecer en `requirements.txt`.

In [1]:
# %pip install -q torch torchvision transformers datasets pillow pandas numpy scikit-learn matplotlib tqdm nltk evaluate accelerate

### **Configuración reproducible**

#### **Semillas, rutas y parámetros**

Esta sección fija los parámetros del experimento. Cambiar estos valores debe quedar justificado en el reporte.

In [2]:
import json
import os
import random
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter, ImageOps
from tqdm.auto import tqdm

SEED = 22514
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

@dataclass
class ExperimentConfig:
    dataset_name: str = "jxie/flickr8k"
    dataset_split: str = "train"
    max_samples: int = 120
    evaluation_samples: int = 100
    caption_samples: int = 24
    error_cases: int = 32
    batch_size: int = 16
    clip_model_name: str = "models_local/clip-vit-base-patch32"  # safetensors local (CVE-2025-32434 / transformers>=5 + torch<2.6)
    blip_model_name: str = "models_local/blip-base"  # safetensors local
    use_blip_captioner: bool = True
    allow_demo_mode: bool = False
    repo_root: str = "."

CONFIG = ExperimentConfig()

ROOT = Path(CONFIG.repo_root).resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUTPUT_FIGURES = ROOT / "outputs" / "figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_METRICS = ROOT / "outputs" / "metrics"
REPORTS = ROOT / "reports"

for folder in [DATA_RAW, DATA_PROCESSED, OUTPUT_FIGURES, OUTPUT_TABLES, OUTPUT_METRICS, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Dispositivo activo: {DEVICE}")
print(json.dumps(asdict(CONFIG), indent=2, ensure_ascii=False))

Dispositivo activo: mps
{
  "dataset_name": "jxie/flickr8k",
  "dataset_split": "train",
  "max_samples": 120,
  "evaluation_samples": 100,
  "caption_samples": 24,
  "error_cases": 32,
  "batch_size": 16,
  "clip_model_name": "models_local/clip-vit-base-patch32",
  "blip_model_name": "models_local/blip-base",
  "use_blip_captioner": true,
  "allow_demo_mode": false,
  "repo_root": "."
}


### **Control del repositorio**

#### **Evidencia mínima de ingeniería experimental**

La evaluación multimodal debe quedar trazable. Esta celda registra versión de Python, versión de PyTorch, disponibilidad de GPU y commit de Git si el repositorio ya fue inicializado.

In [3]:
def get_git_commit() -> str:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception:
        return "repositorio_git_no_detectado"

run_metadata = {
    "semilla": SEED,
    "python": sys.version,
    "torch": torch.__version__,
    "dispositivo": str(DEVICE),
    "cuda_disponible": torch.cuda.is_available(),
    "commit_git": get_git_commit(),
    "configuracion": asdict(CONFIG),
}

metadata_path = OUTPUT_METRICS / "metadata_experimento.json"
metadata_path.write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Metadatos guardados en: {metadata_path}")

Metadatos guardados en: /Users/nielspacheco/Desktop/Classes/UNI/Maestria en ciencias Computacion/IA generativa y multimodal/MCC225-ExamenParcial-Winoground/outputs/metrics/metadata_experimento.json


### **Carga de datos reales**

#### **Dataset imagen texto**

La ruta principal usa Flickr8k desde Hugging Face. Si el estudiante no tiene internet, debe construir un manifiesto local en `data/raw/manifest_local.csv` con columnas:

```text
image_path,caption_1,caption_2,caption_3,caption_4,caption_5
```

El modo de demostración no es válido para entrega final. Solo sirve para verificar que el cuaderno abre.

In [4]:
def detect_image_column(columns: Sequence[str]) -> Optional[str]:
    for name in columns:
        lower = name.lower()
        if lower in {"image", "imagen", "img"} or "image" in lower:
            return name
    return None


def detect_caption_columns(columns: Sequence[str]) -> List[str]:
    caption_columns = []
    for name in columns:
        lower = name.lower()
        if "caption" in lower or "sentence" in lower or "text" in lower or "descripcion" in lower:
            caption_columns.append(name)
    return caption_columns


def save_image_object(image_object, output_path: Path) -> str:
    if isinstance(image_object, Image.Image):
        image = image_object.convert("RGB")
    elif isinstance(image_object, str):
        image = Image.open(image_object).convert("RGB")
    else:
        image = Image.open(image_object).convert("RGB")
    image.save(output_path)
    return str(output_path)


def load_local_manifest(path: Path, max_samples: int) -> pd.DataFrame:
    manifest = pd.read_csv(path)
    required = {"image_path", "caption_1"}
    missing = required.difference(manifest.columns)
    if missing:
        raise ValueError(f"Faltan columnas obligatorias en el manifiesto local: {missing}")
    return manifest.head(max_samples).copy()


def load_real_dataset(config: ExperimentConfig) -> pd.DataFrame:
    local_manifest = DATA_RAW / "manifest_local.csv"
    if local_manifest.exists():
        print("Se usará el manifiesto local.")
        return load_local_manifest(local_manifest, config.max_samples)

    try:
        from datasets import load_dataset
    except Exception as exc:
        raise RuntimeError("Instale datasets o proporcione data/raw/manifest_local.csv") from exc

    try:
        dataset = load_dataset(config.dataset_name, split=config.dataset_split)
    except Exception as exc:
        if config.allow_demo_mode:
            return build_demo_manifest(config.max_samples)
        raise RuntimeError("No se pudo cargar el dataset real. Proporcione un manifiesto local.") from exc

    image_column = detect_image_column(dataset.column_names)
    caption_columns = detect_caption_columns(dataset.column_names)
    if image_column is None or not caption_columns:
        raise ValueError(f"Columnas no reconocidas: {dataset.column_names}")

    records = []
    image_dir = DATA_PROCESSED / "images"
    image_dir.mkdir(parents=True, exist_ok=True)

    limit = min(config.max_samples, len(dataset))
    for index in tqdm(range(limit), desc="Preparando imágenes reales"):
        item = dataset[index]
        image_path = image_dir / f"imagen_{index:05d}.jpg"
        saved_path = save_image_object(item[image_column], image_path)
        record = {"image_id": f"img_{index:05d}", "image_path": saved_path}
        for caption_index, column in enumerate(caption_columns[:5], start=1):
            record[f"caption_{caption_index}"] = str(item[column])
        records.append(record)

    manifest = pd.DataFrame(records)
    manifest_path = DATA_PROCESSED / "manifest_real.csv"
    manifest.to_csv(manifest_path, index=False)
    print(f"Manifiesto real guardado en: {manifest_path}")
    return manifest


def build_demo_manifest(max_samples: int) -> pd.DataFrame:
    raise RuntimeError("El modo de demostración está desactivado para entregas finales.")

manifest = load_real_dataset(CONFIG)
manifest.head()

Se usará el manifiesto local.


  image_id  ...       tag
0   wg_0_0  ...  Relation
1   wg_0_1  ...  Relation
2   wg_1_0  ...  Relation
3   wg_1_1  ...  Relation
4   wg_2_0  ...  Relation

[5 rows x 4 columns]

### **Auditoría del dataset**

#### **Cobertura, duplicados y longitud textual**

Antes de evaluar un modelo, se debe auditar el conjunto de datos. Esto evita confundir desempeño del modelo con problemas de datos.

In [5]:
def get_caption_columns(manifest: pd.DataFrame) -> List[str]:
    return [column for column in manifest.columns if column.startswith("caption_")]


def tokenize_text(text: str) -> List[str]:
    cleaned = "".join(char.lower() if char.isalnum() or char.isspace() else " " for char in str(text))
    return [token for token in cleaned.split() if token]


def audit_dataset(manifest: pd.DataFrame) -> pd.DataFrame:
    caption_columns = get_caption_columns(manifest)
    rows = []
    for _, row in manifest.iterrows():
        captions = [str(row[column]) for column in caption_columns if pd.notna(row[column])]
        lengths = [len(tokenize_text(caption)) for caption in captions]
        rows.append({
            "image_id": row.get("image_id", "sin_id"),
            "num_captions": len(captions),
            "min_len": min(lengths) if lengths else 0,
            "max_len": max(lengths) if lengths else 0,
            "mean_len": float(np.mean(lengths)) if lengths else 0.0,
            "image_exists": Path(row["image_path"]).exists(),
        })
    return pd.DataFrame(rows)

audit = audit_dataset(manifest)
audit_path = OUTPUT_TABLES / "auditoria_dataset.csv"
audit.to_csv(audit_path, index=False)
audit.describe(include="all")

       image_id  num_captions     min_len     max_len    mean_len image_exists
count       120         120.0  120.000000  120.000000  120.000000          120
unique      120           NaN         NaN         NaN         NaN            1
top      wg_0_0           NaN         NaN         NaN         NaN         True
freq          1           NaN         NaN         NaN         NaN          120
mean        NaN           1.0    9.450000    9.450000    9.450000          NaN
std         NaN           0.0    3.528634    3.528634    3.528634          NaN
min         NaN           1.0    5.000000    5.000000    5.000000          NaN
25%         NaN           1.0    7.000000    7.000000    7.000000          NaN
50%         NaN           1.0    9.000000    9.000000    9.000000          NaN
75%         NaN           1.0   11.000000   11.000000   11.000000          NaN
max         NaN           1.0   19.000000   19.000000   19.000000          NaN

### **Visualización de casos**

#### **Revisión cualitativa inicial**

La evaluación debe comenzar con inspección manual. Observe si las imágenes, captions y rutas corresponden.

In [6]:
import matplotlib.pyplot as plt


def show_examples(manifest: pd.DataFrame, n: int = 6) -> None:
    sample = manifest.sample(min(n, len(manifest)), random_state=SEED)
    for _, row in sample.iterrows():
        image = Image.open(row["image_path"]).convert("RGB")
        plt.figure(figsize=(4, 4))
        plt.imshow(image)
        plt.axis("off")
        plt.title(str(row.get("image_id", "imagen")))
        plt.show()
        print("Caption 1:", row.get("caption_1", "sin_caption"))
        print()

show_examples(manifest, n=4)

Caption 1: a plant with some caterpillars

Caption 1: The yellow book is above the blue book and below the red book

Caption 1: the unmasked wrestler hits the masked wrestler

Caption 1: the hurt person is on the left and the helpful person is on the right



### **Evaluación con CLIP**

#### **Embeddings imagen texto y recuperación**

CLIP usa un encoder de imagen y un encoder de texto para proyectar ambas modalidades a un espacio latente común. En esta tarea se evalúa si el texto correcto queda cerca de su imagen.

In [7]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained(CONFIG.clip_model_name).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CONFIG.clip_model_name)
clip_model.eval()

caption_columns = get_caption_columns(manifest)
eval_manifest = manifest.head(CONFIG.evaluation_samples).copy().reset_index(drop=True)
primary_captions = eval_manifest[caption_columns[0]].astype(str).tolist()
image_paths = eval_manifest["image_path"].astype(str).tolist()

print(f"Casos evaluados: {len(eval_manifest)}")

Casos evaluados: 100


In [8]:
@torch.no_grad()
def encode_images(image_paths: Sequence[str], batch_size: int) -> torch.Tensor:
    features = []
    for start in tqdm(range(0, len(image_paths), batch_size), desc="Codificando imágenes"):
        batch_paths = image_paths[start:start + batch_size]
        images = [Image.open(path).convert("RGB") for path in batch_paths]
        inputs = clip_processor(images=images, return_tensors="pt", padding=True).to(DEVICE)
        output = clip_model.get_image_features(**inputs)
        if not torch.is_tensor(output):  # transformers>=5 devuelve un ModelOutput
            output = output.pooler_output
        output = torch.nn.functional.normalize(output, dim=-1)
        features.append(output.cpu())
    return torch.cat(features, dim=0)


@torch.no_grad()
def encode_texts(texts: Sequence[str], batch_size: int) -> torch.Tensor:
    features = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Codificando textos"):
        batch_texts = list(texts[start:start + batch_size])
        inputs = clip_processor(text=batch_texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        output = clip_model.get_text_features(**inputs)
        if not torch.is_tensor(output):  # transformers>=5 devuelve un ModelOutput
            output = output.pooler_output
        output = torch.nn.functional.normalize(output, dim=-1)
        features.append(output.cpu())
    return torch.cat(features, dim=0)


image_embeddings = encode_images(image_paths, CONFIG.batch_size)
text_embeddings = encode_texts(primary_captions, CONFIG.batch_size)
similarity_matrix = image_embeddings @ text_embeddings.T
print(similarity_matrix.shape)

torch.Size([100, 100])


In [9]:
def recall_at_k(similarity: torch.Tensor, k_values: Sequence[int]) -> Dict[str, float]:
    n = similarity.shape[0]
    results = {}
    targets = torch.arange(n)
    for k in k_values:
        top_image_to_text = similarity.topk(k, dim=1).indices
        ok_i2t = (top_image_to_text == targets.unsqueeze(1)).any(dim=1).float().mean().item()

        top_text_to_image = similarity.T.topk(k, dim=1).indices
        ok_t2i = (top_text_to_image == targets.unsqueeze(1)).any(dim=1).float().mean().item()

        results[f"imagen_a_texto_R@{k}"] = ok_i2t
        results[f"texto_a_imagen_R@{k}"] = ok_t2i
    return results


clip_retrieval_metrics = recall_at_k(similarity_matrix, [1, 5, 10])
clip_retrieval_metrics

{'imagen_a_texto_R@1': 0.4399999976158142, 'texto_a_imagen_R@1': 0.38999998569488525, 'imagen_a_texto_R@5': 0.8700000047683716, 'texto_a_imagen_R@5': 0.8199999928474426, 'imagen_a_texto_R@10': 0.9700000286102295, 'texto_a_imagen_R@10': 0.949999988079071}

### **Baseline inicial**

#### **Captions desplazados**

Un resultado serio debe compararse contra un baseline. Aquí se desplazan los captions para romper la correspondencia imagen texto. Si el sistema real no supera claramente este baseline, la evaluación no sostiene la hipótesis de alineamiento.

In [10]:
def shifted_texts(texts: Sequence[str], shift: int = 1) -> List[str]:
    values = list(texts)
    if not values:
        return []
    shift = shift % len(values)
    return values[shift:] + values[:shift]


baseline_captions = shifted_texts(primary_captions, shift=max(1, len(primary_captions) // 3))
baseline_text_embeddings = encode_texts(baseline_captions, CONFIG.batch_size)
baseline_similarity = image_embeddings @ baseline_text_embeddings.T
baseline_metrics = recall_at_k(baseline_similarity, [1, 5, 10])

comparison_metrics = pd.DataFrame([
    {"sistema": "CLIP_con_pares_reales", **clip_retrieval_metrics},
    {"sistema": "baseline_captions_desplazados", **baseline_metrics},
])
comparison_path = OUTPUT_TABLES / "comparacion_retrieval.csv"
comparison_metrics.to_csv(comparison_path, index=False)
comparison_metrics

                         sistema  ...  texto_a_imagen_R@10
0          CLIP_con_pares_reales  ...                 0.95
1  baseline_captions_desplazados  ...                 0.04

[2 rows x 7 columns]

### **CLIPScore simplificado**

#### **Fidelidad global imagen texto**

El CLIPScore simplificado usa la similitud coseno entre embedding de imagen y embedding de texto. No reemplaza evaluación humana, pero ayuda a ordenar casos para inspección manual.

In [11]:
def compute_clipscore(similarity: torch.Tensor) -> pd.DataFrame:
    diagonal = similarity.diag().numpy()
    scores = pd.DataFrame({
        "image_id": eval_manifest["image_id"].tolist(),
        "image_path": eval_manifest["image_path"].tolist(),
        "caption": primary_captions,
        "clipscore_simplificado": diagonal,
    })
    return scores.sort_values("clipscore_simplificado", ascending=True).reset_index(drop=True)


clipscore_table = compute_clipscore(similarity_matrix)
clipscore_path = OUTPUT_TABLES / "clipscore_casos.csv"
clipscore_table.to_csv(clipscore_path, index=False)
clipscore_table.head(10)

  image_id  ... clipscore_simplificado
0  wg_39_0  ...               0.177985
1  wg_39_1  ...               0.191658
2  wg_49_1  ...               0.204436
3   wg_3_0  ...               0.210928
4   wg_4_1  ...               0.212528
5  wg_24_0  ...               0.216025
6  wg_11_0  ...               0.217250
7   wg_4_0  ...               0.217879
8  wg_21_0  ...               0.220727
9  wg_35_1  ...               0.221398

[10 rows x 4 columns]

### **Métricas léxicas para captions**

#### **BLEU simple, ROUGE L y cobertura**

Estas métricas son parciales. Sirven para comparar texto generado contra referencias, pero pueden ignorar errores visuales. El estudiante debe discutir esta limitación.

In [12]:
def ngrams(tokens: Sequence[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[index:index + n]) for index in range(len(tokens) - n + 1)]


def simple_bleu(candidate: str, references: Sequence[str], n: int = 4) -> float:
    candidate_tokens = tokenize_text(candidate)
    if not candidate_tokens:
        return 0.0
    scores = []
    for size in range(1, n + 1):
        candidate_ngrams = ngrams(candidate_tokens, size)
        if not candidate_ngrams:
            scores.append(0.0)
            continue
        reference_set = set()
        for reference in references:
            reference_set.update(ngrams(tokenize_text(reference), size))
        matches = sum(1 for gram in candidate_ngrams if gram in reference_set)
        scores.append(matches / max(1, len(candidate_ngrams)))
    return float(np.mean(scores))


def longest_common_subsequence(a: Sequence[str], b: Sequence[str]) -> int:
    table = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i, j] = table[i - 1, j - 1] + 1
            else:
                table[i, j] = max(table[i - 1, j], table[i, j - 1])
    return int(table[len(a), len(b)])


def rouge_l(candidate: str, references: Sequence[str]) -> float:
    candidate_tokens = tokenize_text(candidate)
    if not candidate_tokens:
        return 0.0
    best_score = 0.0
    for reference in references:
        reference_tokens = tokenize_text(reference)
        if not reference_tokens:
            continue
        lcs = longest_common_subsequence(candidate_tokens, reference_tokens)
        precision = lcs / max(1, len(candidate_tokens))
        recall = lcs / max(1, len(reference_tokens))
        if precision + recall == 0:
            score = 0.0
        else:
            score = 2 * precision * recall / (precision + recall)
        best_score = max(best_score, score)
    return float(best_score)


def lexical_coverage(candidate: str, references: Sequence[str]) -> float:
    candidate_set = set(tokenize_text(candidate))
    reference_set = set()
    for reference in references:
        reference_set.update(tokenize_text(reference))
    if not reference_set:
        return 0.0
    return len(candidate_set.intersection(reference_set)) / len(reference_set)

### **Generación opcional con BLIP**

#### **Captioner real**

Esta sección es obligatoria si el estudiante tiene GPU o CPU suficiente. Si no se ejecuta, debe justificarse y reemplazarse por captions generados previamente, guardados en `data/processed/captions_generados.csv`.

In [13]:
def generate_blip_captions(image_paths: Sequence[str], batch_size: int) -> List[str]:
    from transformers import BlipForConditionalGeneration, BlipProcessor

    processor = BlipProcessor.from_pretrained(CONFIG.blip_model_name)
    model = BlipForConditionalGeneration.from_pretrained(CONFIG.blip_model_name).to(DEVICE)
    model.eval()
    captions = []
    for start in tqdm(range(0, len(image_paths), batch_size), desc="Generando captions con BLIP"):
        batch_paths = image_paths[start:start + batch_size]
        images = [Image.open(path).convert("RGB") for path in batch_paths]
        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=32)
        decoded = processor.batch_decode(generated, skip_special_tokens=True)
        captions.extend([text.strip() for text in decoded])
    return captions


caption_output_path = DATA_PROCESSED / "captions_generados.csv"

if CONFIG.use_blip_captioner:
    caption_subset = eval_manifest.head(CONFIG.caption_samples).copy()
    generated_captions = generate_blip_captions(caption_subset["image_path"].tolist(), CONFIG.batch_size)
    caption_subset["caption_generado"] = generated_captions
    caption_subset.to_csv(caption_output_path, index=False)
    print(f"Captions generados guardados en: {caption_output_path}")
elif caption_output_path.exists():
    caption_subset = pd.read_csv(caption_output_path)
    print("Se cargaron captions generados previamente.")
else:
    caption_subset = eval_manifest.head(CONFIG.caption_samples).copy()
    caption_subset["caption_generado"] = caption_subset["caption_1"].astype(str)
    print("Advertencia: se usan captions de referencia como marcador temporal. Para la entrega final se debe generar o cargar caption_generado real.")

caption_subset.head()

Captions generados guardados en: /Users/nielspacheco/Desktop/Classes/UNI/Maestria en ciencias Computacion/IA generativa y multimodal/MCC225-ExamenParcial-Winoground/data/processed/captions_generados.csv


  image_id  ...                              caption_generado
0   wg_0_0  ...              an old man kissing a little girl
1   wg_0_1  ...  a woman and a child kissing each other woman
2   wg_1_0  ...            a man and woman hugging each other
3   wg_1_1  ...         a man and woman standing in a kitchen
4   wg_2_0  ...             a wrestler wrestling in a stadium

[5 rows x 5 columns]

In [14]:
def evaluate_generated_captions(table: pd.DataFrame) -> pd.DataFrame:
    caption_columns = get_caption_columns(table)
    rows = []
    for _, row in table.iterrows():
        references = [str(row[column]) for column in caption_columns
                      if column != "caption_generado" and pd.notna(row[column])]  # evita fuga: excluye el candidato de las referencias
        candidate = str(row["caption_generado"])
        rows.append({
            "image_id": row.get("image_id", "sin_id"),
            "caption_generado": candidate,
            "bleu_simple": simple_bleu(candidate, references),
            "rouge_l": rouge_l(candidate, references),
            "cobertura_lexica": lexical_coverage(candidate, references),
        })
    return pd.DataFrame(rows)


caption_metrics = evaluate_generated_captions(caption_subset)
caption_metrics_path = OUTPUT_TABLES / "metricas_captions.csv"
caption_metrics.to_csv(caption_metrics_path, index=False)
caption_metrics.describe()

       bleu_simple    rouge_l  cobertura_lexica
count    24.000000  24.000000         24.000000
mean      0.072719   0.190827          0.263355
std       0.058099   0.157406          0.234425
min       0.000000   0.000000          0.000000
25%       0.023438   0.065217          0.057692
50%       0.066477   0.153846          0.200000
75%       0.127537   0.291209          0.425000
max       0.166667   0.470588          0.800000

### **Simulación CapFilt**

#### **Captioner y filter**

BLIP usa bootstrapping para trabajar con textos web ruidosos. Aquí se simula esa idea con tres candidatos por imagen: un caption de referencia, un caption generado y un caption desplazado que actúa como ruido.

In [15]:
def build_capfilt_candidates(table: pd.DataFrame) -> pd.DataFrame:
    shifted = shifted_texts(table["caption_1"].astype(str).tolist(), shift=max(1, len(table) // 4))
    records = []
    for index, row in table.reset_index(drop=True).iterrows():
        records.append({
            "image_id": row["image_id"],
            "image_path": row["image_path"],
            "candidate_type": "humano_referencia",
            "candidate_caption": str(row["caption_1"]),
            "expected_label": 1,
        })
        records.append({
            "image_id": row["image_id"],
            "image_path": row["image_path"],
            "candidate_type": "captioner_sintetico",
            "candidate_caption": str(row.get("caption_generado", row["caption_1"])),
            "expected_label": 1,
        })
        records.append({
            "image_id": row["image_id"],
            "image_path": row["image_path"],
            "candidate_type": "texto_web_ruidoso",
            "candidate_caption": shifted[index],
            "expected_label": 0,
        })
    return pd.DataFrame(records)


capfilt_candidates = build_capfilt_candidates(caption_subset)
capfilt_text_embeddings = encode_texts(capfilt_candidates["candidate_caption"].tolist(), CONFIG.batch_size)
capfilt_image_embeddings = encode_images(capfilt_candidates["image_path"].tolist(), CONFIG.batch_size)
capfilt_candidates["clipscore_simplificado"] = (capfilt_image_embeddings * capfilt_text_embeddings).sum(dim=1).numpy()
capfilt_candidates.head()

  image_id  ... clipscore_simplificado
0   wg_0_0  ...               0.298788
1   wg_0_0  ...               0.302544
2   wg_0_0  ...               0.219102
3   wg_0_1  ...               0.300145
4   wg_0_1  ...               0.267660

[5 rows x 6 columns]

In [16]:
def evaluate_filter_threshold(candidates: pd.DataFrame, threshold: float) -> Dict[str, float]:
    predicted = (candidates["clipscore_simplificado"] >= threshold).astype(int)
    expected = candidates["expected_label"].astype(int)
    tp = int(((predicted == 1) & (expected == 1)).sum())
    fp = int(((predicted == 1) & (expected == 0)).sum())
    fn = int(((predicted == 0) & (expected == 1)).sum())
    tn = int(((predicted == 0) & (expected == 0)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    accuracy = (tp + tn) / max(1, tp + tn + fp + fn)
    return {
        "umbral": threshold,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


thresholds = np.linspace(
    float(capfilt_candidates["clipscore_simplificado"].min()),
    float(capfilt_candidates["clipscore_simplificado"].max()),
    num=21,
)
filter_results = pd.DataFrame([evaluate_filter_threshold(capfilt_candidates, float(value)) for value in thresholds])
filter_results_path = OUTPUT_TABLES / "capfilt_umbral.csv"
filter_results.to_csv(filter_results_path, index=False)
filter_results.sort_values("accuracy", ascending=False).head()

      umbral  precision    recall  accuracy  tp  fp  fn  tn
7   0.210407   0.886792  0.979167  0.902778  47   6   1  18
6   0.199969   0.842105  1.000000  0.875000  48   9   0  15
8   0.220845   0.914894  0.895833  0.875000  43   4   5  20
9   0.231283   0.931818  0.854167  0.861111  41   3   7  21
10  0.241721   0.974359  0.791667  0.847222  38   1  10  23

### **Ablación visual**

#### **Robustez ante degradación**

Esta sección mide si el alineamiento imagen texto cambia cuando la imagen pierde información. El estudiante debe comparar imagen original, baja resolución, desenfoque, escala de grises y recorte central.

In [17]:
def degrade_image(image: Image.Image, mode: str) -> Image.Image:
    image = image.convert("RGB")
    if mode == "original":
        return image
    if mode == "baja_resolucion":
        small = image.resize((max(1, image.width // 4), max(1, image.height // 4)))
        return small.resize(image.size)
    if mode == "desenfoque":
        return image.filter(ImageFilter.GaussianBlur(radius=3))
    if mode == "grises":
        return ImageOps.grayscale(image).convert("RGB")
    if mode == "recorte_central":
        width, height = image.size
        margin_w = width // 5
        margin_h = height // 5
        cropped = image.crop((margin_w, margin_h, width - margin_w, height - margin_h))
        return cropped.resize(image.size)
    raise ValueError(f"Modo de degradación no reconocido: {mode}")


def build_degraded_dataset(manifest: pd.DataFrame, modes: Sequence[str]) -> pd.DataFrame:
    records = []
    degraded_dir = DATA_PROCESSED / "degradadas"
    degraded_dir.mkdir(parents=True, exist_ok=True)
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Creando imágenes degradadas"):
        image = Image.open(row["image_path"]).convert("RGB")
        for mode in modes:
            output_path = degraded_dir / f"{row['image_id']}_{mode}.jpg"
            degrade_image(image, mode).save(output_path)
            records.append({
                "image_id": row["image_id"],
                "mode": mode,
                "image_path": str(output_path),
                "caption_1": row["caption_1"],
            })
    return pd.DataFrame(records)


ablacion_modes = ["original", "baja_resolucion", "desenfoque", "grises", "recorte_central"]
ablacion_manifest = build_degraded_dataset(eval_manifest.head(80), ablacion_modes)
ablacion_manifest.head()

  image_id  ...                            caption_1
0   wg_0_0  ...  an old person kisses a young person
1   wg_0_0  ...  an old person kisses a young person
2   wg_0_0  ...  an old person kisses a young person
3   wg_0_0  ...  an old person kisses a young person
4   wg_0_0  ...  an old person kisses a young person

[5 rows x 4 columns]

In [18]:
def evaluate_visual_degradation(ablacion_manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for mode, group in ablacion_manifest.groupby("mode"):
        image_features = encode_images(group["image_path"].tolist(), CONFIG.batch_size)
        text_features = encode_texts(group["caption_1"].astype(str).tolist(), CONFIG.batch_size)
        similarity = image_features @ text_features.T
        metrics = recall_at_k(similarity, [1, 5, 10])
        metrics["modo"] = mode
        metrics["clipscore_promedio"] = float(similarity.diag().mean().item())
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values("modo")


ablacion_results = evaluate_visual_degradation(ablacion_manifest)
ablacion_results_path = OUTPUT_TABLES / "ablacion_visual.csv"
ablacion_results.to_csv(ablacion_results_path, index=False)
ablacion_results

   imagen_a_texto_R@1  texto_a_imagen_R@1  ...             modo  clipscore_promedio
0              0.4500              0.3750  ...  baja_resolucion            0.267459
1              0.4625              0.4000  ...       desenfoque            0.266969
2              0.4375              0.3875  ...           grises            0.259197
3              0.4750              0.4250  ...         original            0.269653
4              0.4500              0.3750  ...  recorte_central            0.261993

[5 rows x 8 columns]

### **Análisis de errores**

#### **Taxonomía obligatoria**

El análisis debe clasificar errores con estas categorías:

```text
objeto, acción, conteo, relación espacial, OCR, atributo, alucinación, sesgo, respuesta vaga, otro
```

Cada estudiante debe anotar manualmente al menos 30 casos. Se deben incluir fallos y aciertos, no solo ejemplos favorables.

In [19]:
ERROR_CATEGORIES = [
    "objeto",
    "acción",
    "conteo",
    "relación espacial",
    "OCR",
    "atributo",
    "alucinación",
    "sesgo",
    "respuesta vaga",
    "otro",
]


def prepare_error_sheet(scores: pd.DataFrame, n_cases: int) -> pd.DataFrame:
    worst = scores.head(n_cases // 2).copy()
    random_cases = scores.sample(n=min(n_cases - len(worst), len(scores)), random_state=SEED).copy()
    selected = pd.concat([worst, random_cases], ignore_index=True).drop_duplicates("image_id")
    selected["categoria_error"] = "pendiente"
    selected["severidad_1_a_5"] = "pendiente"
    selected["evidencia_visual"] = "pendiente"
    selected["comentario_tecnico"] = "pendiente"
    return selected


error_sheet = prepare_error_sheet(clipscore_table, CONFIG.error_cases)
error_sheet_path = OUTPUT_TABLES / "plantilla_analisis_errores.csv"
error_sheet.to_csv(error_sheet_path, index=False)
print(f"Plantilla de análisis de errores guardada en: {error_sheet_path}")
error_sheet.head()

Plantilla de análisis de errores guardada en: /Users/nielspacheco/Desktop/Classes/UNI/Maestria en ciencias Computacion/IA generativa y multimodal/MCC225-ExamenParcial-Winoground/outputs/tables/plantilla_analisis_errores.csv


  image_id  ... comentario_tecnico
0  wg_39_0  ...          pendiente
1  wg_39_1  ...          pendiente
2  wg_49_1  ...          pendiente
3   wg_3_0  ...          pendiente
4   wg_4_1  ...          pendiente

[5 rows x 8 columns]

In [20]:
def show_error_case(error_table: pd.DataFrame, index: int) -> None:
    row = error_table.iloc[index]
    image = Image.open(row["image_path"]).convert("RGB")
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(str(row["image_id"]))
    plt.show()
    print("Caption evaluado:", row["caption"])
    print("CLIPScore simplificado:", row["clipscore_simplificado"])
    print("Categorías permitidas:", ", ".join(ERROR_CATEGORIES))

show_error_case(error_sheet, 0)

Caption evaluado: the hurt person is on the left and the helpful person is on the right
CLIPScore simplificado: 0.1779853
Categorías permitidas: objeto, acción, conteo, relación espacial, OCR, atributo, alucinación, sesgo, respuesta vaga, otro


### **Perturbación textual**

#### **Sensibilidad a negación, conteo y atributos**

El objetivo es detectar si el modelo distingue captions correctos de captions alterados. Se modifican atributos, conteos, objetos o negaciones de forma controlada.

In [21]:
def build_text_perturbations(caption: str) -> List[Tuple[str, str]]:
    tokens = tokenize_text(caption)
    original = " ".join(tokens)
    perturbations = [("original", original)]
    perturbations.append(("negacion", f"no {original}"))
    perturbations.append(("conteo", f"dos {original}"))
    perturbations.append(("atributo", f"{original} de color rojo"))
    perturbations.append(("objeto_externo", f"{original} con un avion visible"))
    return perturbations


def evaluate_text_perturbations(manifest: pd.DataFrame, n: int = 40) -> pd.DataFrame:
    records = []
    subset = manifest.head(n).copy()
    for _, row in subset.iterrows():
        for perturbation_type, text in build_text_perturbations(str(row["caption_1"])):
            records.append({
                "image_id": row["image_id"],
                "image_path": row["image_path"],
                "tipo_perturbacion": perturbation_type,
                "texto": text,
            })
    table = pd.DataFrame(records)
    image_features = encode_images(table["image_path"].tolist(), CONFIG.batch_size)
    text_features = encode_texts(table["texto"].tolist(), CONFIG.batch_size)
    table["clipscore_simplificado"] = (image_features * text_features).sum(dim=1).numpy()
    return table


perturbation_results = evaluate_text_perturbations(eval_manifest, n=40)
perturbation_path = OUTPUT_TABLES / "perturbacion_textual.csv"
perturbation_results.to_csv(perturbation_path, index=False)
perturbation_results.groupby("tipo_perturbacion")["clipscore_simplificado"].describe()

                   count      mean       std  ...       50%       75%       max
tipo_perturbacion                             ...                              
atributo            40.0  0.260790  0.035568  ...  0.265577  0.290145  0.314664
conteo              40.0  0.258052  0.031043  ...  0.261451  0.284887  0.317214
negacion            40.0  0.264438  0.032352  ...  0.270826  0.286896  0.315789
objeto_externo      40.0  0.270290  0.031711  ...  0.274883  0.294906  0.320586
original            40.0  0.272691  0.031471  ...  0.279529  0.299146  0.324014

[5 rows x 8 columns]

### **Figuras para la exposición**

#### **Matriz de similitud y comparación de métricas**

La exposición debe incluir al menos dos figuras generadas desde el cuaderno, no imágenes copiadas de internet.

In [22]:
def plot_similarity_matrix(similarity: torch.Tensor, output_path: Path, max_items: int = 40) -> None:
    values = similarity[:max_items, :max_items].numpy()
    plt.figure(figsize=(7, 6))
    plt.imshow(values)
    plt.colorbar()
    plt.title("Matriz de similitud imagen texto")
    plt.xlabel("Texto")
    plt.ylabel("Imagen")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()


def plot_metric_bars(table: pd.DataFrame, output_path: Path) -> None:
    metric_columns = [column for column in table.columns if column != "sistema"]
    melted = table.melt(id_vars="sistema", value_vars=metric_columns, var_name="metrica", value_name="valor")
    plt.figure(figsize=(10, 5))
    for index, system in enumerate(melted["sistema"].unique()):
        subset = melted[melted["sistema"] == system]
        positions = np.arange(len(subset)) + index * 0.35
        plt.bar(positions, subset["valor"], width=0.35, label=system)
    plt.xticks(np.arange(len(metric_columns)) + 0.18, metric_columns, rotation=45, ha="right")
    plt.ylabel("Valor")
    plt.title("Comparación contra baseline")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()


plot_similarity_matrix(similarity_matrix, OUTPUT_FIGURES / "matriz_similitud_clip.png")
plot_metric_bars(comparison_metrics, OUTPUT_FIGURES / "comparacion_baseline.png")

### **Consolidación de resultados**

#### **Archivo único de métricas**

Todo resultado usado en la exposición debe quedar guardado en disco.

In [23]:
all_metrics = {
    "retrieval_clip": clip_retrieval_metrics,
    "retrieval_baseline": baseline_metrics,
    "caption_metrics_mean": caption_metrics.select_dtypes(include=[np.number]).mean().to_dict(),
    "capfilt_best": filter_results.sort_values("accuracy", ascending=False).head(1).to_dict(orient="records")[0],
    "ablacion_visual": ablacion_results.to_dict(orient="records"),
}

metrics_path = OUTPUT_METRICS / "metricas_finales.json"
metrics_path.write_text(json.dumps(all_metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Métricas finales guardadas en: {metrics_path}")

Métricas finales guardadas en: /Users/nielspacheco/Desktop/Classes/UNI/Maestria en ciencias Computacion/IA generativa y multimodal/MCC225-ExamenParcial-Winoground/outputs/metrics/metricas_finales.json


### **Reporte técnico**

#### **Plantilla obligatoria**

El estudiante debe completar `reports/reporte_exposicion_2.md`. El reporte debe ser breve, técnico y reproducible.

In [24]:
def write_report_template(path: Path) -> None:
    text = """
### Reporte individual de evaluación multimodal

#### Datos del estudiante

Nombre:
Repositorio:
Commit final:
Fecha:

#### Tarea definida

Explica qué sistema evaluastes, qué entrada recibe y qué salida produce.

#### Dataset

Describe el dataset usado, número de imágenes, número de captions, filtros aplicados y partición evaluada.

#### Modelos evaluados

Indica modelos, checkpoints, parámetros relevantes y configuración de inferencia.

#### Baselines

Explica el baseline usado y por qué es una comparación mínima razonable.

#### Métricas

Reporte Recall@K, CLIPScore simplificado, métricas léxicas, resultados de CapFilt y resultados de ablación visual.

#### Análisis de errores

Incluye tabla de errores con categorías: objeto, acción, conteo, relación espacial, OCR, atributo, alucinación, sesgo, respuesta vaga y otro.

#### Discusión de limitaciones

Explica qué no capturan las métricas, dónde falla el modelo y qué supuestos pueden afectar la validez del experimento.

#### Conexión conceptual

Relaciona los resultados con BERT, VisualBERT, UNITER, ViLT, BLIP, BLIP-2 y LLaVA.

#### Conclusión

Resume si el sistema muestra alineamiento multimodal robusto o solo desempeño parcial.
""".strip()
    path.write_text(text, encoding="utf-8")


report_path = REPORTS / "reporte_exposicion_2.md"
if not report_path.exists():
    write_report_template(report_path)
print(f"Plantilla de reporte disponible en: {report_path}")

Plantilla de reporte disponible en: /Users/nielspacheco/Desktop/Classes/UNI/Maestria en ciencias Computacion/IA generativa y multimodal/MCC225-ExamenParcial-Winoground/reports/reporte_exposicion_2.md


### **Cierre conceptual**

#### **Relación con los modelos estudiados**

BERT introduce la gramática de tokens, embeddings y atención bidireccional. VisualBERT, UNITER y ViLT extienden esa gramática a tokens visuales. BLIP agrega generación y limpieza de datos mediante bootstrapping. CoCa combina alineamiento contrastivo y captioning. BLIP-2 conecta visión congelada con LLMs congelados mediante Q-Former. LLaVA lleva la imagen al espacio del LLM para seguir instrucciones visuales.

Este cuaderno evalúa esa línea desde el punto de vista experimental: alineamiento, recuperación, generación, filtrado, ablación, error, reproducibilidad y límites. Esa es la base mínima para una exposición académica fuerte.